In [ ]:
#bloque de imports de librerias DE LOS METODOS
import pandas as pd

from pgmpy.models import BayesianNetwork #la terminal me dijo que esta obsoleto
from pgmpy.estimators import HillClimbSearch, ExhaustiveSearch, BIC, K2

#los siguientes import son para 
from pgmpy.estimators import MaximumLikelihoodEstimator  # esto nos servira para estimar los parametros, tal como se enseña en la documentacion
from pgmpy.models import DiscreteBayesianNetwork


#import pgmpy.estimators
#print(dir(pgmpy.estimators))     # esto sirve para poder ver que clases y funciones tiene el modulo de estimadores de pgmpy, lo cual es util para saber que opciones tenemos


In [ ]:
#imports de librerias para graficar

import networkx as nx
import matplotlib.pyplot as plt


In [ ]:
import pgmpy
print(pgmpy.__version__)

In [ ]:
#cargado de datos y seleccion

#creamos un array con los nombres de las variables (columnas del archivo)
#como el archivo no viene con cabeceras este array servira para definirlas
# las definiciones de cada nombre de columna estan en la documentacion y se deben añadir en el informe
col_name = ['poisonuos', 'cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor', 
           'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color', 
           'stalk-shape', 'stalk-root', 'stalk-surface-above-ring', 
           'stalk-surface-below-ring', 'stalk-color-above-ring', 
           'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number', 
           'ring-type', 'spore-print-color', 'population', 'habitat']


doc_completo = pd.read_csv('./mushroom/agaricus-lepiota.data', header=None, names=col_name)

HC_doc = doc_completo.copy() # aqui tenemos las 23 columnas

ES_col_name = ['poisonuos', 'cap-color', 'odor', 'stalk-root', 'veil-type', 'spore-print-color', 'habitat'  ]

#el copy() es mejor que hacer un script donde se guarden solo las columnas segun el nombre, asi es mas facil
ES_doc = doc_completo[ES_col_name].copy() # asi el doc solo tendra las columnas con ese nombre, en vez de meter todas en 5 o 7 columnas



In [ ]:
hc = HillClimbSearch(HC_doc) #aplicamos el metodo HC para encontrar la estructura de la red bayesiana, el resultado se guarda en la variable hc
#este metodo se basa en una busqueda local que comienza con una estructura vacia
#  y va añadiendo, eliminando o revirtiendo arcos para mejorar la puntuacion del modelo segun el criterio de BIC

#obtenemos el mejor modelo encontrado por HC utilizando el criterio de BIC para evaluar la calidad del modelo
HC_mejor_modelo = hc.estimate(scoring_method=BIC(HC_doc))
#por ejemplo un resultado podria ser: (cap-shape, cap-color) lo que indica que hay un arco dirigido desde la variable cap-shape 
# hacia la variable cap-color, lo que sugiere que el color del sombrero puede depender de su forma

#los arcos encontrados por HC representan las relaciones de dependencia entre las variables del conjunto de datos
print("Arcos encontrados: ")
print(HC_mejor_modelo.edges())


In [ ]:
#definir el método de puntuación con tus datos
puntuacion = K2(ES_doc)

#pasar la puntuación AL CONSTRUCTOR del objeto ExhaustiveSearch
est = ExhaustiveSearch(ES_doc, scoring_method=puntuacion)

#ejecutar el .estimate()
ES_mejor_modelo = est.estimate()

print("Arcos encontrados por Búsqueda Exhaustiva:")
print(best_model.edges())

In [ ]:
modelo_a_graficar = mejor_modelo_hc 

# Convertimos los arcos aprendidos en un Grafo Dirigido de NetworkX
G_hc = nx.DiGraph(modelo_a_graficar.edges())

# --- ESTILO Y DISEÑO (AQUÍ ESTÁ LO "BONITO") ---
plt.figure(figsize=(16, 10)) # Tamaño grande para que se lea bien

# Diseño: Usamos 'shell_layout' para organizar los nodos en círculos concéntricos
# Esto ordena mucho la vista cuando hay muchas variables.
pos_hc = nx.shell_layout(G_hc)

# 1. Dibujar los Nodos (Círculos)
nx.draw_networkx_nodes(G_hc, pos_hc, 
                       node_size=4000,          # Tamaño grande
                       node_color='#a1d99b',    # Verde suave (estilo hongos)
                       edgecolors='black',      # Borde negro fino
                       linewidths=1.5)

# 2. Dibujar los Arcos (Flechas)
nx.draw_networkx_edges(G_hc, pos_hc, 
                       edgelist=G_hc.edges(),
                       edge_color='#636363',    # Gris oscuro
                       arrowsize=25,            # Flechas grandes y visibles
                       arrowstyle='->',        # Estilo de flecha limpia
                       width=2.0)               # Grosor de la línea

# 3. Dibujar las Etiquetas (Nombres de variables)
nx.draw_networkx_labels(G_hc, pos_hc, 
                        font_size=12, 
                        font_family='sans-serif', 
                        font_weight='bold')

plt.title("Estructura de Red Bayesiana: Hill-Climbing Search", fontsize=20, fontweight='bold', pad=20)
plt.axis('off') # Ocultar los ejes cartesianos
plt.tight_layout() # Ajustar márgenes automáticamente
plt.show()

In [ ]:
modelo_es_a_graficar = best_model 

G_es = nx.DiGraph(modelo_es_a_graficar.edges())

# Si el modelo no encontró arcos (grafo vacío), avisamos para no dar error
if G_es.number_of_edges() == 0:
    print("El modelo de Búsqueda Exhaustiva no encontró ninguna conexión significativa (arcos).")
    # Dibujamos solo los nodos sueltos para que no quede en blanco
    G_es.add_nodes_from(modelo_es_a_graficar.nodes()) 

plt.figure(figsize=(12, 8)) # Un poco más pequeño porque hay menos nodos

# Diseño: Usamos 'spring_layout' que separa los nodos como si fueran imanes.
# Es ideal para ver estructuras claras en grafos pequeños.
pos_es = nx.spring_layout(G_es, k=1.0, iterations=50)

# 1. Dibujar los Nodos
nx.draw_networkx_nodes(G_es, pos_es, 
                       node_size=5000,          # Nodos aún más grandes
                       node_color='#9ecae1',    # Azul suave
                       edgecolors='black', 
                       linewidths=1.5)

# 2. Dibujar los Arcos (con K2 Score)
nx.draw_networkx_edges(G_es, pos_es, 
                       edgelist=G_es.edges(),
                       edge_color='#636363', 
                       arrowsize=30,            # Flechas extra grandes
                       arrowstyle='->',
                       arrows = True,
                       width=2.5)               # Líneas más gruesas

# 3. Dibujar las Etiquetas
nx.draw_networkx_labels(G_es, pos_es, 
                        font_size=14, 
                        font_family='sans-serif', 
                        font_weight='bold')

# --- FINALIZAR ---
plt.title("Estructura de Red Bayesiana: Exhaustive Search (K2 Score)", fontsize=20, fontweight='bold', pad=20)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 1. Crear un objeto de dibujo con la estructura aprendida
# (Suponiendo que tu modelo se llama 'mejor_modelo_hc')
model_graph = nx.DiGraph(mejor_modelo_hc.edges())

# 2. Configurar el diseño (layout) y el tamaño de la figura
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(model_graph, k=0.5) # Ajusta 'k' para separar más los nodos

# 3. Dibujar nodos, flechas y etiquetas
nx.draw(model_graph, pos, with_labels=True, node_size=3000, 
        node_color="skyblue", font_size=10, font_weight="bold", 
        arrowsize=20, edge_color="gray")

plt.title("Estructura de la Red Bayesiana (Hill-Climbing)")
plt.show()

In [ ]:
HC_modelo_final = DiscreteBayesianNetwork(HC_mejor_modelo.edges()) # tengo que ver bien que wea hace esto porque solo copie y pegue esto de la documentacion
ES_modelo_final = DiscreteBayesianNetwork(ES_mejor_modelo.edges())

#aqui hacemos la estimacion de parametros
HC_modelo_final.fit(HC_doc, estimator=MaximumLikelihoodEstimator)
ES_modelo_final.fit(ES_doc, estimator=MaximumLikelihoodEstimator)


HC_modelo_final.cpds
ES_modelo_final.cpds
